# Wildfire Risk Dashboard: Data Preparation Pipeline
This notebook is used to convert data from the raw file (df_final_model.csv) into a JSON database to be used on the web dashboard.

In [1]:
# Import necessary libraries for data manipulation (pandas), JSON processing, and OS operations.
import pandas as pd
import json
import os

# 1. Define file paths
# Set the paths for the input raw CSV data and the output JSON destination.
INPUT_CSV = '../Dataset/df_final_model.csv'
OUTPUT_JSON = '../../wildfire risk visualization/data/district_points_data.json'

print("Loading data...")
# Read the CSV file into a pandas DataFrame. 'low_memory=False' is used to ensure accurate data type inference across large files.
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Successfully loaded: {len(df)} rows")

Loading data...
Successfully loaded: 68447 rows


## 2. Point-based Extraction Process
We will select the latest month's data for each district and extract the actual coordinates to display them as cluster points on the map.

In [2]:
# Import necessary libraries for data manipulation and numerical operations.
import pandas as pd
import numpy as np

# 1. Select the latest month's data
# Find the maximum value in the 'month' column to identify the most recent period.
latest_month = df['month'].max()
# Filter the DataFrame to keep only the records from the latest month and create an independent copy.
df_latest = df[df['month'] == latest_month].copy()

# (Assuming that if you have used the model for prediction, you will get the pred_risk_prob column)
# df_latest['pred_risk_prob'] = model.predict_proba(X_scaled)[:, 1] * 100

# If there's no model yet, use your previous Baseline Risk simulation formula for now (multiply by 100 to make it a percentage)
# Calculate a baseline risk probability using a weighted combination of temperature and inverse soil moisture.
df_latest['pred_risk_prob'] = ((df_latest['temp'] / 40) + ((1 - df_latest['soil_moisture']) * 0.5)) * 100

db_points = {}

# Define the maximum number of points to display per district (can be adjusted based on web performance)
max_points_per_district = 50 

print("Preparing point data for the web...")

# 2. Use Groupby to process district by district
# Iterate through each province and district group in the dataset.
for (prov, dist), group in df_latest.groupby(['NAME_1', 'NAME_2']):
    
    # Initialize the province dictionary if it doesn't exist yet.
    if prov not in db_points: 
        db_points[prov] = {}
        
    db_points[prov][dist] = []
    
    # 3. Sampling representative points to ensure distribution across the district
    if len(group) > max_points_per_district:
        # Select the 50 points with the highest risk values in that district
        # Extract the top 'max_points_per_district' rows based on their predicted risk probability.
        sampled_group = group.nlargest(max_points_per_district, 'pred_risk_prob')
    else:
        sampled_group = group
    
    # 4. Extract the sampled point data into JSON format
    # Iterate over the rows of the sampled group to extract coordinates and metrics into a dictionary.
    for _, row in sampled_group.iterrows():
        db_points[prov][dist].append({
            'lat': float(row['LATITUDE']),
            'lng': float(row['LONGITUDE']),
            'risk_prob': float(row['pred_risk_prob']), 
            'temp': float(row['temp']),
            'ndvi': float(row['ndvi']),
            # Check if the CONFIDENCE column exists before using it
            'confidence': int(row['CONFIDENCE']) if 'CONFIDENCE' in row and pd.notnull(row['CONFIDENCE']) else 0
        })

print(f"Processing complete: Total {len(db_points)} provinces")

Preparing point data for the web...
Processing complete: Total 19 provinces


## 3. Saving the Results for Web Usage

In [3]:
# Open the output JSON file in write mode with UTF-8 encoding to support international characters.
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    # Serialize the db_points dictionary into JSON format and write it to the file. 
    # 'ensure_ascii=False' allows non-ASCII characters, and 'indent=2' formats it for readability.
    json.dump(db_points, f, ensure_ascii=False, indent=2)

print(f"✅ File saved to: {OUTPUT_JSON}")
print("You can now run the web page and click Predict to view these points immediately!")

✅ File saved to: ../../wildfire risk visualization/data/district_points_data.json
You can now run the web page and click Predict to view these points immediately!
